# 09 - Multi-Book Lead-Time Tests and Dashboard Comparison

Purpose:
- apply one lead-time and false-alarm template to a **parallel multi-book evaluation**;
- compare whether the same Gold alarm leads deterioration in Brent, WTI, and Henry Hub proxy books;
- export a common dashboard view for each book plus cross-book comparison tables.

Core design:
- Gold alarm timing is fixed from prior notebooks;
- each book's event set is derived from its own return, volatility, and drawdown path;
- comparisons are made at common parameter settings rather than by re-tuning each book separately.


## Reader Orientation

Notebook 08 built the parallel multi-book evaluation. This notebook asks the next operational question: **does the same Gold alarm provide useful lead time across those books?**

The evaluation stays deliberately symmetric across books so that the comparison remains defensible.

Prerequisite outputs: this notebook expects `08_RiskBookVaRStress.ipynb` to have produced `data/processed/riskbook_var_multi.parquet` and the Step 08 CSV outputs in `outputs/step08_riskbook_var_stress/`.


In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from statsmodels.stats.proportion import proportion_confint

ROOT = Path.cwd()
PROCESSED_DIR = ROOT / 'data' / 'processed'
OUTPUT_DIR = ROOT / 'outputs' / 'step09_lead_time_dashboard'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

In [ ]:
riskbook_path = PROCESSED_DIR / 'riskbook_var_multi.parquet'
stress_path = ROOT / 'outputs' / 'step08_riskbook_var_stress' / 'stress_scenarios_multi.csv'

riskbook_multi = pd.read_parquet(riskbook_path)
stress_scenarios = pd.read_csv(stress_path) if stress_path.exists() else pd.DataFrame()

required_cols = ['book', 'R_book', 'nav', 'drawdown', 'hs_var_return', 'var_breach', 'realized_vol_20d', 'gold_alarm', 'alarm_score', 'dashboard_state']
missing_cols = [col for col in required_cols if col not in riskbook_multi.columns]
if missing_cols:
    raise ValueError(f'Missing expected Step 08 columns: {missing_cols}')

print('Loaded multi-book riskbook:', riskbook_path, riskbook_multi.shape)
print('Books:', sorted(riskbook_multi['book'].unique().tolist()))

## Parameter Rationale

These settings stay fixed across books:
- `LOOKBACK_DAYS = 30`
- `COOLDOWN_DAYS = 5`
- `DRAWDOWN_EVENT_LEVEL = -0.05`
- `VOL_Z_THRESHOLD = 2.0`

The purpose is comparison, not per-book optimization.


In [ ]:
LOOKBACK_DAYS = 30
COOLDOWN_DAYS = 5
DRAWDOWN_EVENT_LEVEL = -0.05
VOL_Z_THRESHOLD = 2.0

## Helper Functions

In [ ]:
def trailing_zscore(series: pd.Series, window: int) -> pd.Series:
    mean = series.rolling(window).mean().shift(1)
    std = series.rolling(window).std().shift(1)
    return (series - mean) / std


def event_starts(flag: pd.Series) -> pd.DatetimeIndex:
    flag = flag.fillna(0).astype(int)
    starts = flag.eq(1) & flag.shift(1, fill_value=0).eq(0)
    return pd.DatetimeIndex(flag.index[starts])


def apply_cooldown(flag: pd.Series, cooldown_days: int) -> pd.Series:
    cooled = pd.Series(0, index=flag.index, dtype=int)
    last_fire = None
    for date, value in flag.fillna(0).astype(int).items():
        if value != 1:
            continue
        if last_fire is None or (date - last_fire).days > cooldown_days:
            cooled.loc[date] = 1
            last_fire = date
    return cooled


def match_leads(alarm_dates: pd.DatetimeIndex, event_dates: pd.DatetimeIndex, lookback_days: int) -> pd.DataFrame:
    rows = []
    for event_date in event_dates:
        candidates = alarm_dates[
            (alarm_dates <= event_date)
            & (alarm_dates >= event_date - pd.Timedelta(days=lookback_days))
        ]
        if len(candidates) == 0:
            rows.append({'event_date': event_date, 'matched': False, 'alarm_date': pd.NaT, 'lead_days': np.nan})
        else:
            alarm_date = candidates[-1]
            rows.append({'event_date': event_date, 'matched': True, 'alarm_date': alarm_date, 'lead_days': (event_date - alarm_date).days})
    return pd.DataFrame(rows)


def evaluate_book_leads(book_df: pd.DataFrame) -> tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame, pd.DataFrame]:
    analysis = book_df.copy().sort_index()
    analysis['cooled_gold_alarm'] = apply_cooldown(analysis['gold_alarm'], COOLDOWN_DAYS)
    analysis['portfolio_vol_z'] = trailing_zscore(analysis['realized_vol_20d'], 252)
    analysis['portfolio_vol_spike'] = (analysis['portfolio_vol_z'] > VOL_Z_THRESHOLD).astype(int)
    analysis['drawdown_event'] = (analysis['drawdown'] <= DRAWDOWN_EVENT_LEVEL).astype(int)

    alarm_dates = event_starts(analysis['cooled_gold_alarm'])
    event_families = {
        'var_breach': event_starts(analysis['var_breach']),
        'portfolio_vol_spike': event_starts(analysis['portfolio_vol_spike']),
        'drawdown_event': event_starts(analysis['drawdown_event']),
    }

    lead_tables = []
    for family, dates in event_families.items():
        table = match_leads(alarm_dates, dates, LOOKBACK_DAYS)
        table['event_family'] = family
        lead_tables.append(table)
    lead_time_table = pd.concat(lead_tables, ignore_index=True) if lead_tables else pd.DataFrame()

    lead_summary = (
        lead_time_table.groupby('event_family')
        .agg(
            event_count=('event_date', 'count'),
            matched_count=('matched', 'sum'),
            match_rate=('matched', 'mean'),
            avg_lead_days=('lead_days', 'mean'),
            median_lead_days=('lead_days', 'median'),
        )
        .reset_index()
    )
    if not lead_summary.empty:
        ci_rows = []
        for _, row in lead_summary.iterrows():
            n = int(row['event_count'])
            k = int(row['matched_count'])
            lo, hi = proportion_confint(k, n, alpha=0.05, method='wilson') if n > 0 else (np.nan, np.nan)
            ci_rows.append({'event_family': row['event_family'], 'ci_low_95': lo, 'ci_high_95': hi})
        lead_summary = lead_summary.merge(pd.DataFrame(ci_rows), on='event_family')

    all_events = pd.DatetimeIndex(sorted(set().union(*[set(v) for v in event_families.values()])))
    false_alarm_rows = []
    for alarm_date in alarm_dates:
        future_events = all_events[(all_events >= alarm_date) & (all_events <= alarm_date + pd.Timedelta(days=LOOKBACK_DAYS))]
        false_alarm_rows.append({
            'alarm_date': alarm_date,
            'followed_by_event': len(future_events) > 0,
            'next_event_date': future_events[0] if len(future_events) else pd.NaT,
        })
    false_alarm_table = pd.DataFrame(false_alarm_rows)
    false_alarm_summary = pd.DataFrame([{
        'alarm_count': len(false_alarm_table),
        'false_alarm_count': int((~false_alarm_table['followed_by_event']).sum()) if len(false_alarm_table) else 0,
        'false_alarm_rate': float((~false_alarm_table['followed_by_event']).mean()) if len(false_alarm_table) else np.nan,
    }])

    return analysis, lead_time_table, lead_summary, false_alarm_summary

## Run Lead-Time Template Across Books

In [ ]:
analysis_by_book = {}
lead_tables = []
lead_summaries = []
false_alarm_summaries = []
dashboard_rows = []

for book_name in sorted(riskbook_multi['book'].unique()):
    book_df = riskbook_multi.loc[riskbook_multi['book'] == book_name].copy()
    analysis, lead_table, lead_summary, false_alarm_summary = evaluate_book_leads(book_df)
    analysis_by_book[book_name] = analysis

    if not lead_table.empty:
        lead_table['book'] = book_name
        lead_tables.append(lead_table)
    if not lead_summary.empty:
        lead_summary['book'] = book_name
        lead_summaries.append(lead_summary)
    false_alarm_summary['book'] = book_name
    false_alarm_summaries.append(false_alarm_summary)

    latest = analysis.iloc[-1].copy()
    dashboard_rows.append({
        'book': book_name,
        'latest_date': analysis.index[-1],
        'dashboard_state': latest['dashboard_state'],
        'latest_alarm_score': latest['alarm_score'],
        'latest_nav': latest['nav'],
        'latest_drawdown': latest['drawdown'],
        'latest_var': latest['hs_var_return'],
        'latest_realized_vol_20d': latest['realized_vol_20d'],
        'latest_var_breach': int(latest['var_breach']),
    })

book_lead_time_table = pd.concat(lead_tables, ignore_index=True) if lead_tables else pd.DataFrame()
book_lead_time_comparison = pd.concat(lead_summaries, ignore_index=True) if lead_summaries else pd.DataFrame()
book_false_alarm_summary = pd.concat(false_alarm_summaries, ignore_index=True)
book_dashboard_snapshot = pd.DataFrame(dashboard_rows)

book_lead_time_comparison.head(12)

## Dashboard Comparison Plot

This chart puts the same four diagnostics side by side across books using each book's own evaluation sample.


In [ ]:
fig, axes = plt.subplots(len(analysis_by_book), 4, figsize=(18, 4 * len(analysis_by_book)), sharex=False)
if len(analysis_by_book) == 1:
    axes = np.array([axes])

for row_idx, (book_name, analysis) in enumerate(analysis_by_book.items()):
    axes[row_idx, 0].plot(analysis.index, analysis['alarm_score'], linewidth=0.9)
    axes[row_idx, 0].set_title(f'{book_name}: Gold alarm score')

    analysis[['R_book', 'hs_var_return']].plot(ax=axes[row_idx, 1], linewidth=0.8)
    axes[row_idx, 1].set_title(f'{book_name}: Book return vs VaR')

    analysis['drawdown'].plot(ax=axes[row_idx, 2], linewidth=0.9)
    axes[row_idx, 2].axhline(DRAWDOWN_EVENT_LEVEL, color='red', linestyle='--', linewidth=0.8)
    axes[row_idx, 2].set_title(f'{book_name}: Drawdown')

    analysis['portfolio_vol_z'].plot(ax=axes[row_idx, 3], linewidth=0.9)
    axes[row_idx, 3].axhline(VOL_Z_THRESHOLD, color='red', linestyle='--', linewidth=0.8)
    axes[row_idx, 3].set_title(f'{book_name}: Portfolio vol z-score')

plt.tight_layout()

## Cross-Book Readout

In [ ]:
summary_table = book_lead_time_comparison.merge(book_false_alarm_summary, on='book', how='left')
summary_table[['book', 'event_family', 'event_count', 'matched_count', 'match_rate', 'avg_lead_days', 'false_alarm_rate']]

## Save Multi-Book Outputs

In [ ]:
book_lead_time_table.to_csv(OUTPUT_DIR / 'book_lead_time_table.csv', index=False)
book_lead_time_comparison.to_csv(OUTPUT_DIR / 'book_lead_time_comparison.csv', index=False)
book_false_alarm_summary.to_csv(OUTPUT_DIR / 'book_false_alarm_summary.csv', index=False)
book_dashboard_snapshot.to_csv(OUTPUT_DIR / 'book_dashboard_snapshot.csv', index=False)

all_dashboards = []
for book_name, analysis in analysis_by_book.items():
    dashboard = analysis[[
        'book', 'R_book', 'nav', 'drawdown', 'hs_var_return', 'hs_es_return', 'var_breach',
        'realized_vol_20d', 'portfolio_vol_z', 'portfolio_vol_spike', 'gold_alarm',
        'cooled_gold_alarm', 'alarm_score', 'dashboard_state'
    ]].copy()
    dashboard['recommended_action'] = np.select(
        [dashboard['dashboard_state'].eq('Red'), dashboard['dashboard_state'].eq('Amber')],
        ['Run stress tests and review VaR calibration', 'Monitor and inspect gold relationship components'],
        default='Normal monitoring',
    )
    all_dashboards.append(dashboard)

multi_dashboard = pd.concat(all_dashboards).sort_index()
multi_dashboard.to_csv(OUTPUT_DIR / 'dashboard_metrics_multi.csv')

print('Saved Step 09 multi-book outputs to:', OUTPUT_DIR)